In [ ]:
using Bessels, MPSDynamics, ChaosTools, ProgressBars
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/config/requirements.jl");

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

#### Caracterization of the circuit #####

ω_a = 3.8533*2*pi
ω_b_0 = 7.0515*2*pi

Lamb_shift = 0.05 #Lamb shift in GHz


φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv = Z_b*ω_b_0
E_Cb = C_b_inv/2

E_J = 37.12*2*pi


##Drive


ω_d=ω_b_0#+Lamb_shift*2*pi # Drive frequency
ω_p=2ω_a-ω_d

ϵ_0 = 0.5
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=10.12*g2

buffer_phase = 0.0
pump_phase = 0.0;

##Env

κ_b=1/10.4; # 1/ns from Leon's paper
κ_2 = 4*g2^2/κ_b
κ_a = 0#3e-3*κ_2 #from Leon's paper

In [ ]:
t_max=20/κ_b
re_a_0, im_a_0 = 9, -4
ϕ_a_0, q_a_0=sqrt(2*Z_a)*re_a_0, sqrt(2/Z_a)*im_a_0
ϕ_b_0, q_b_0=0.0, 0.0

system = SystemATS(ω_a, ω_b_0, E_J, E_Ca, E_Cb, ϕ_a_0, q_a_0, ϕ_b_0, q_b_0)
drive = DriveTwoModes(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
environment = MeanField(κ_a, κ_b)

ϕ_CL_a, ϕ_CL_b, q_CL_a, q_CL_b, t_list_CL = simulator(t_max, system, drive, environment);

In [ ]:
p1,p2 = plot_mode(ϕ_CL_a, q_CL_a, t_list_CL, ω_a, Z_a)

plot_perso(p1,p2,layout=(1,2),
xlabel=L"\Re(a)", ylabel=L"\Im(a)",
title=["Resonator a" "Resonator a RF"],
legend=false)

In [ ]:

t_min = 0.0
dt = 1 / (50 * ω_a) # Time step for the simulation

system_prepared = prepare(system)
drive_prepared = prepare(drive)
environment_prepared = prepare(system,environment)

u_dot_0, u_0 = get_init(system_prepared, environment_prepared)
u_0_new = vcat(u_0,u_dot_0)

p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

problem = CoupledODEs(equation_1ODE!, u_0_new, p);

In [ ]:
Lyap_conv=[]
t_max = 10/κ_b
for i in range(start = 0, stop = 60, step = 2)
    push!(Lyap_conv,lyapunov(problem, t_max, Δt=dt; d0 = 1e-7, d0_upper = 1e-4, Ttr = i/κ_b))
end

In [ ]:
Lyap_conv_2=[]
t_max = 40/κ_b
for i in range(start = 0, stop = 60, step = 2)
    push!(Lyap_conv_2,lyapunov(problem, t_max, Δt=dt; d0 = 1e-7, d0_upper = 1e-4, Ttr = i/κ_b))
end

In [ ]:
plot(range(start = 0, stop = 60, step = 2), Lyap_conv, label="Computation over "*L"10/\kappa_b")
plot!(range(start = 0, stop = 60, step = 2), Lyap_conv_2, label="Computation over "*L"40/\kappa_b")
scatter!((0,Lyap_conv[1]), m=:x, label = "Previous working point")
scatter!((30,Lyap_conv[16]), m=:x, label = "New working point")
plot!(ylabel = "Lyapunov exponent", xlabel = "Waited time before computation of the exponent "*L"\kappa_bt_W")